# M2 — the single Model 2: every condition field, one checkpoint

Two earlier runs each solved half the problem and broke the other half.

**B3** trained on every row carrying any condition field. Only 26.1% of them record a
temperature, so `?` was the answer that minimised the loss: it abstained on 86.5% of test
records and reached 15.7% within ±10 °C.

**T2** trained only on rows that do record one. Temperature jumped to 54.6%, but 71% of the
chemistry was gone: outside that subset its catalyst fell to 9.6% against B3's 29.3%.

Neither is deliverable. A route needs a temperature and a solvent every time — ORD's silence
means *not logged*, not *not needed*, unlike the catalyst, which really is absent from most
reactions and where `?` is the true answer.

## The schema prefix

So the mandatory fields are never written as `?`, and rows that lack them are never dropped.
A row supervises what it can, and its input carries a code naming the layout:

| code | rows | target |
|---|---:|---|
| `[ST]` | 26.1% | `reagents\|solvent\|catalyst\|temperature` |
| `[S]` | 63.1% | `reagents\|solvent\|catalyst` |
| `[T]` | 2.7% | `reagents\|catalyst\|temperature` |
| `[0]` | 8.1% | `reagents\|catalyst` |

At inference the evaluator always asks `[ST]`, so the model must produce both mandatory
fields — it has never seen a `[ST]` example where either was `?`. This is T5's own task-prefix
scheme (Raffel et al., 2020), used for chemistry by T5Chem (Lu & Zhang, JCIM 2022) to put
five reaction tasks into one checkpoint.

## The corpus and the budget

`kuzmenkoiryna/retro-planner-ord-conditions-mixed`: 219,922 training rows — all 139,922 that
record a temperature plus 80,000 that do not (63.6% / 36.4%), leak-checked against the test
(5,084 rows removed). The mix is a budget decision. Temperature supervision costs passes over
a small pool; breadth costs passes over a large one. At 3 epochs this buys 420k temperature
examples and 660k of everything else, against B3's 229k / 832k and T2's 1,049k / 1,049k.

**Budget: 6 GPU hours left in the week**, and the cascade run still has to fit in it. Training
is capped at 200 minutes, evaluation takes about 30.

**Base:** t5-small, set by B2 (beats CompoundT5 on solvent p<0.0001 and catalyst p=0.0001).

**Before running:** Settings → **Internet** on, **GPU T4 x2** on. Run as **Save & Run All (Commit)**.

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available(), "| devices:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"  Device {i}:", torch.cuda.get_device_name(i))

In [ ]:
import os
if not os.path.isdir("retro-planner"):
    !git clone https://github.com/oleh-kuzmenko/retro-planner.git
%cd retro-planner

In [ ]:
%pip install -q -e ".[local-models,indexing]"

In [ ]:
import glob, json, collections

train_file = next(glob.iglob("/kaggle/input/**/conditions_train.jsonl", recursive=True))
val_file = next(glob.iglob("/kaggle/input/**/conditions_val.jsonl", recursive=True))
test_file = next(glob.iglob("/kaggle/input/**/conditions_test_clean.jsonl", recursive=True))
for path in (train_file, val_file, test_file):
    print(path, sum(1 for _ in open(path)), "rows")

rows = [json.loads(line) for line in open(train_file)]
def filled(field):
    return sum(1 for r in rows if r.get(field) not in (None, "")) / len(rows)
print("\nfield coverage in train:", {f: f"{filled(f):.1%}" for f in
      ("reagents", "solvent", "catalyst", "temperature_celsius")})

# The roles split must have travelled with the data, not been re-derived here.
assert "reagents" in rows[0] and "full_reactants_smiles" in rows[0], "dataset is the pre-roles one"

base_model = "t5-small"
learning_rate = 5e-4
condition_fields = "reagents,solvent,catalyst,temperature_celsius"
always_fields = "solvent,temperature_celsius"
output_dir = "/kaggle/working/model2_conditions_mixed"
time_budget_minutes = 200

In [ ]:
import os

os.makedirs(output_dir, exist_ok=True)
log_path = f"{output_dir}/train.log"

!torchrun --nproc_per_node=2 scripts/train_conditions_model.py \
    --base-model "{base_model}" \
    --train-file "{train_file}" \
    --val-file "{val_file}" \
    --output-dir "{output_dir}" \
    --local-work-dir /kaggle/temp/local_model2_mixed \
    --target-format compact \
    --condition-fields "{condition_fields}" \
    --always-fields "{always_fields}" \
    --max-source-length 256 \
    --max-target-length 256 \
    --learning-rate {learning_rate} \
    --num-train-epochs 3 \
    --time-budget-minutes {time_budget_minutes} \
    > "{log_path}" 2>&1
print("training done; tail of log:")
!tail -5 "{log_path}"

In [ ]:
# The schema counts printed here are the check that the prefix scheme actually engaged:
# four codes must appear, and ST must be the largest single-mandatory-field group.
!grep -E "schema codes|Predicting|Train examples|new character token" "{log_path}"

import json
from transformers import AutoTokenizer

saved_tokenizer = AutoTokenizer.from_pretrained(f"{output_dir}/final")
embedding_rows = json.load(open(f"{output_dir}/final/config.json"))["vocab_size"]
print("tokenizer length:", len(saved_tokenizer), "| embedding rows:", embedding_rows)
assert len(saved_tokenizer) == embedding_rows, "tokenizer and embedding matrix disagree"
marker = json.load(open(f"{output_dir}/final/conditions_format.json"))
print("format marker:", marker)
assert marker["always_fields"] == always_fields.split(","), "marker lost the mandatory fields"

In [ ]:
import json

state = json.load(open(f"{output_dir}/latest_checkpoint/trainer_state.json"))
points = [(h["epoch"], h["eval_loss"]) for h in state["log_history"] if "eval_loss" in h]
for epoch, loss in points[:: max(1, len(points) // 12)]:
    print(f"  epoch {epoch:5.2f}  eval_loss {loss:.4f}")
best_epoch = min(points, key=lambda p: p[1])[0]
print(f"  best {state.get('best_metric')} at epoch {best_epoch:.2f} of {points[-1][0]:.2f} reached")

In [ ]:
# The evaluator reads `always_fields` from the marker and prompts every record with [ST],
# so the numbers below are what a chemist would get asking for a full recipe every time.
!python scripts/evaluate_conditions_model_topk.py \
    --model-dir "{output_dir}/final" \
    --test-file "{test_file}" \
    --num-beams 10 --batch-size 32 --device cuda \
    --max-source-length 256 --max-target-length 256 \
    --output "/kaggle/working/M2_conditions_mixed_clean_topk.json"

In [ ]:
import json
data = json.load(open("/kaggle/working/M2_conditions_mixed_clean_topk.json"))
summary = data["summary"]
print(json.dumps(summary, indent=2))

# Read against the two runs this one has to replace. B3 is the breadth reference,
# T2 the temperature reference; the point of the prefix scheme is to lose neither.
reference = {
    "reagents_exact_match_top5": ("B3 0.215", "T2 0.184"),
    "solvent_exact_match_top5": ("B3 0.338", "T2 0.269"),
    "catalyst_exact_match_top5": ("B3 0.288", "T2 0.176"),
    "temperature_celsius_within_tol_top5": ("B3 0.157", "T2 0.546"),
}
print("\nfield                                   this run   previous")
for key, (b3, t2) in reference.items():
    print(f"{key:38} {summary.get(key, 0):.3f}   {b3} / {t2}")

# A model that never abstains on a mandatory field is only honest if it also stops
# guessing where the reference has nothing to check against.
records = data["records"]
no_temp = [r for r in records if r["reference"].get("temperature_celsius") in (None, "")]
print(f"\nrecords with no reference temperature: {len(no_temp)} "
      f"(the model answers on all of them by construction; correctness unmeasurable)")